# Testes de lambda do FAGTB

Este notebook usa os mesmos splits salvos pelo fluxo principal e treina o FAGTB via `modelo_fagtb`, importado de `func_aux`.

A ideia é variar apenas `lambda_fagtb`, mantendo os hiperparâmetros padrão definidos dentro de `modelo_fagtb`.

In [55]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import json
import joblib
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

import func_aux
importlib.reload(func_aux)
from func_aux import *

Arquivo exportado: func_aux.py


In [56]:
# ============================================================
# Pastas
# ============================================================

pasta_tratado = Path("datasets/tratado")
pasta_resultados = Path("resultados")

pasta_modelos = pasta_resultados / "modelos"
pasta_splits = pasta_resultados / "splits"

pasta_testes_fagtb = pasta_resultados / "testes_lambda_fagtb"
pasta_figuras_fagtb = pasta_testes_fagtb / "figuras"

for pasta in [
    pasta_modelos,
    pasta_splits,
    pasta_testes_fagtb,
    pasta_figuras_fagtb
]:
    pasta.mkdir(parents=True, exist_ok=True)

with open("configs_datasets.json", "r", encoding="utf-8") as f:
    configs = json.load(f)

arquivos = list(pasta_tratado.glob("*.csv"))

sobrescrever = True
gerar_grafico = True
mostrar_grafico = False
salvar_modelo_final_teste = False

In [57]:
# ============================================================
# Valores de lambda a testar
# ============================================================

lambdas = [
    0.001,
    0.003,
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
    3.0,
    10.0
]

# Regra de escolha:
# escolhe o lambda imediatamente anterior à primeira queda brusca
# da métrica de desempenho.
metrica_despencar = "balanced_accuracy"
queda_brusca_minima = 0.05
metrica_justica = "equalized_odds"

# Split interno usado apenas para escolher lambda dentro de X_train.
test_size_validacao = 0.25
random_state = 42

In [58]:
# ============================================================
# Carregamento dos splits salvos pelo fluxo principal
# ============================================================

def remover_colunas_indice(df):
    return df.loc[:, ~df.columns.str.contains("^Unnamed")]


def carregar_splits_salvos(nome_dataset, pasta_splits="resultados/splits"):
    pasta_splits = Path(pasta_splits)

    caminho_X_train = pasta_splits / f"{nome_dataset}_X_train.csv"
    caminho_X_test = pasta_splits / f"{nome_dataset}_X_test.csv"
    caminho_y_train = pasta_splits / f"{nome_dataset}_y_train.csv"
    caminho_y_test = pasta_splits / f"{nome_dataset}_y_test.csv"

    arquivos_necessarios = [
        caminho_X_train,
        caminho_X_test,
        caminho_y_train,
        caminho_y_test
    ]

    for caminho in arquivos_necessarios:
        if not caminho.exists():
            raise FileNotFoundError(
                f"Split não encontrado: {caminho}. "
                "Rode primeiro o fluxo principal para gerar os splits."
            )

    X_train = remover_colunas_indice(pd.read_csv(caminho_X_train))
    X_test = remover_colunas_indice(pd.read_csv(caminho_X_test))
    y_train = remover_colunas_indice(pd.read_csv(caminho_y_train)).iloc[:, 0]
    y_test = remover_colunas_indice(pd.read_csv(caminho_y_test)).iloc[:, 0]

    X_train = X_train.reset_index(drop=True)
    X_test = X_test.reset_index(drop=True)
    y_train = y_train.astype(int).reset_index(drop=True)
    y_test = y_test.astype(int).reset_index(drop=True)

    return X_train, X_test, y_train, y_test

In [59]:
# ============================================================
# Métricas auxiliares
# ============================================================

def _safe_div(num, den):
    return np.nan if den == 0 else num / den


def taxas_por_grupo(y_true, y_pred, sensitive, grupo):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    sensitive = np.asarray(sensitive)

    mask = sensitive == grupo

    yt = y_true[mask]
    yp = y_pred[mask]

    tp = np.sum((yt == 1) & (yp == 1))
    fn = np.sum((yt == 1) & (yp == 0))
    fp = np.sum((yt == 0) & (yp == 1))
    tn = np.sum((yt == 0) & (yp == 0))

    tpr = _safe_div(tp, tp + fn)
    fpr = _safe_div(fp, fp + tn)

    return tpr, fpr


def equal_opportunity_diff(
    y_true,
    y_pred,
    sensitive,
    grupo_privilegiado,
    grupo_desprivilegiado
):
    tpr_priv, _ = taxas_por_grupo(
        y_true,
        y_pred,
        sensitive,
        grupo_privilegiado
    )

    tpr_des, _ = taxas_por_grupo(
        y_true,
        y_pred,
        sensitive,
        grupo_desprivilegiado
    )

    return abs(tpr_priv - tpr_des)


def equalized_odds_diff(
    y_true,
    y_pred,
    sensitive,
    grupo_privilegiado,
    grupo_desprivilegiado
):
    tpr_priv, fpr_priv = taxas_por_grupo(
        y_true,
        y_pred,
        sensitive,
        grupo_privilegiado
    )

    tpr_des, fpr_des = taxas_por_grupo(
        y_true,
        y_pred,
        sensitive,
        grupo_desprivilegiado
    )

    return 0.5 * (
        abs(tpr_priv - tpr_des) +
        abs(fpr_priv - fpr_des)
    )


def demographic_parity_diff(
    y_pred,
    sensitive,
    grupo_privilegiado,
    grupo_desprivilegiado
):
    y_pred = np.asarray(y_pred).astype(int)
    sensitive = np.asarray(sensitive)

    taxa_priv = y_pred[sensitive == grupo_privilegiado].mean()
    taxa_des = y_pred[sensitive == grupo_desprivilegiado].mean()

    return abs(taxa_priv - taxa_des)


def p_rule_percent(
    y_pred,
    sensitive,
    grupo_privilegiado,
    grupo_desprivilegiado
):
    y_pred = np.asarray(y_pred).astype(int)
    sensitive = np.asarray(sensitive)

    taxa_priv = y_pred[sensitive == grupo_privilegiado].mean()
    taxa_des = y_pred[sensitive == grupo_desprivilegiado].mean()

    if taxa_priv == 0 or taxa_des == 0:
        return 0.0

    return min(
        taxa_des / taxa_priv,
        taxa_priv / taxa_des
    ) * 100


def avaliar_modelo(
    y_true,
    y_pred,
    sensitive,
    grupo_privilegiado,
    grupo_desprivilegiado
):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),

        "equal_opportunity": equal_opportunity_diff(
            y_true,
            y_pred,
            sensitive,
            grupo_privilegiado,
            grupo_desprivilegiado
        ),

        "equalized_odds": equalized_odds_diff(
            y_true,
            y_pred,
            sensitive,
            grupo_privilegiado,
            grupo_desprivilegiado
        ),

        "demographic_parity": demographic_parity_diff(
            y_pred,
            sensitive,
            grupo_privilegiado,
            grupo_desprivilegiado
        ),

        "p_rule": p_rule_percent(
            y_pred,
            sensitive,
            grupo_privilegiado,
            grupo_desprivilegiado
        )
    }

In [60]:
# ============================================================
# Escolha automática do lambda
# ============================================================

def escolher_lambda_antes_despencar(
    df_val,
    metrica_desempenho="balanced_accuracy",
    metrica_justica="equalized_odds",
    queda_brusca_minima=0.05
):
    df_ord = df_val.copy()
    df_ord = df_ord.sort_values("lambda").reset_index(drop=True)

    df_ord["queda_desempenho"] = (
        df_ord[metrica_desempenho].shift(1) -
        df_ord[metrica_desempenho]
    )

    colapsos = df_ord[
        df_ord["queda_desempenho"] >= queda_brusca_minima
    ]

    if not colapsos.empty:
        idx_colapso = colapsos.index[0]

        if idx_colapso == 0:
            melhor_idx = 0
        else:
            melhor_idx = idx_colapso - 1

        melhor_lambda = df_ord.loc[melhor_idx, "lambda"]

        info_escolha = {
            "criterio": "antes_da_primeira_queda_brusca",
            "metrica_desempenho": metrica_desempenho,
            "metrica_justica": metrica_justica,
            "queda_brusca_minima": queda_brusca_minima,

            "lambda_antes_queda": melhor_lambda,
            "lambda_onde_caiu": df_ord.loc[idx_colapso, "lambda"],
            "queda_detectada": df_ord.loc[idx_colapso, "queda_desempenho"],

            "valor_antes_queda": df_ord.loc[melhor_idx, metrica_desempenho],
            "valor_depois_queda": df_ord.loc[idx_colapso, metrica_desempenho]
        }

        return melhor_lambda, info_escolha, df_ord

    candidatos = df_ord.sort_values(
        by=[metrica_justica, metrica_desempenho, "accuracy"],
        ascending=[True, False, False]
    )

    melhor_lambda = candidatos.iloc[0]["lambda"]

    info_escolha = {
        "criterio": "menor_justica_sem_queda_brusca_detectada",
        "metrica_desempenho": metrica_desempenho,
        "metrica_justica": metrica_justica,
        "queda_brusca_minima": queda_brusca_minima,

        "lambda_antes_queda": None,
        "lambda_onde_caiu": None,
        "queda_detectada": None,

        "valor_antes_queda": None,
        "valor_depois_queda": None
    }

    return melhor_lambda, info_escolha, df_ord

In [61]:
# ============================================================
# Gráfico único com dois eixos Y
# ============================================================

def plotar_tradeoff_lambda_duplo_eixo(
    df_resultados,
    melhor_lambda=None,
    info_escolha=None,
    nome_experimento="fagtb",
    cenario="aware",
    pasta_figuras="resultados/testes_lambda_fagtb/figuras",
    salvar_figura=True,
    mostrar_figura=False,
    dpi=200
):
    df_val = df_resultados[df_resultados["split"] == "validacao"].copy()
    df_val = df_val.sort_values("lambda")

    pasta_figuras = Path(pasta_figuras)
    pasta_figuras.mkdir(parents=True, exist_ok=True)

    fig, ax1 = plt.subplots(figsize=(11, 5.5))

    ax1.plot(
        df_val["lambda"],
        df_val["accuracy"],
        marker="o",
        label="Accuracy"
    )

    ax1.plot(
        df_val["lambda"],
        df_val["f1"],
        marker="o",
        label="F1"
    )

    ax1.plot(
        df_val["lambda"],
        df_val["balanced_accuracy"],
        marker="o",
        label="Balanced Accuracy"
    )

    ax1.set_xlabel("Lambda")
    ax1.set_ylabel("Desempenho tradicional")
    ax1.set_xscale("symlog", linthresh=0.001)
    ax1.grid(True)

    ax2 = ax1.twinx()

    ax2.plot(
        df_val["lambda"],
        df_val["equalized_odds"],
        marker="s",
        linestyle="--",
        label="Equalized Odds"
    )

    ax2.plot(
        df_val["lambda"],
        df_val["equal_opportunity"],
        marker="s",
        linestyle="--",
        label="Equal Opportunity"
    )

    ax2.set_ylabel("Diferença de justiça")

    if melhor_lambda is not None:
        ax1.axvline(
            x=melhor_lambda,
            linestyle=":",
            linewidth=2,
            label=f"Lambda escolhido = {melhor_lambda}"
        )

    if info_escolha is not None:
        lambda_onde_caiu = info_escolha.get("lambda_onde_caiu")

        if lambda_onde_caiu is not None:
            ax1.axvline(
                x=lambda_onde_caiu,
                linestyle=":",
                linewidth=1,
                label=f"Queda detectada = {lambda_onde_caiu}"
            )

    linhas_1, labels_1 = ax1.get_legend_handles_labels()
    linhas_2, labels_2 = ax2.get_legend_handles_labels()

    ax1.legend(
        linhas_1 + linhas_2,
        labels_1 + labels_2,
        loc="center right"
    )

    titulo = f"Trade-off desempenho vs justiça - {nome_experimento} - {cenario}"
    plt.title(titulo)

    fig.tight_layout()

    caminho_figura = None

    if salvar_figura:
        nome_arquivo = f"{nome_experimento}_{cenario}_tradeoff_lambda.png"
        caminho_figura = pasta_figuras / nome_arquivo

        fig.savefig(
            caminho_figura,
            dpi=dpi,
            bbox_inches="tight"
        )

        print(f"Figura salva em: {caminho_figura}")

    if mostrar_figura:
        plt.show()

    plt.close(fig)

    return caminho_figura

In [62]:
# ============================================================
# Função principal: testar lambdas
# ============================================================

def testar_lambdas_fagtb(
    X_train,
    X_test,
    y_train,
    y_test,
    atributo_sensivel,
    grupo_privilegiado,
    grupo_desprivilegiado,
    lambdas,
    random_state=42,
    test_size_validacao=0.25,
    metrica_despencar="balanced_accuracy",
    queda_brusca_minima=0.05,
    metrica_justica="equalized_odds",
    pasta_saida="resultados/testes_lambda_fagtb",
    nome_experimento="fagtb",
    usar_atributo_sensivel_no_modelo=True,
    salvar_modelo_final=False,
    gerar_grafico=True,
    mostrar_grafico=False
):
    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    X_train = X_train.copy().reset_index(drop=True)
    X_test = X_test.copy().reset_index(drop=True)

    y_train = pd.Series(y_train).astype(int).reset_index(drop=True)
    y_test = pd.Series(y_test).astype(int).reset_index(drop=True)

    s_train = X_train[atributo_sensivel].astype(int).reset_index(drop=True)
    s_test = X_test[atributo_sensivel].astype(int).reset_index(drop=True)

    X_fit, X_val, y_fit, y_val, s_fit, s_val = train_test_split(
        X_train,
        y_train,
        s_train,
        test_size=test_size_validacao,
        random_state=random_state,
        stratify=y_train
    )

    if usar_atributo_sensivel_no_modelo:
        X_fit_model = X_fit.copy()
        X_val_model = X_val.copy()
        X_train_model = X_train.copy()
        X_test_model = X_test.copy()
        sufixo = "aware"
    else:
        X_fit_model = X_fit.drop(columns=[atributo_sensivel], errors="ignore")
        X_val_model = X_val.drop(columns=[atributo_sensivel], errors="ignore")
        X_train_model = X_train.drop(columns=[atributo_sensivel], errors="ignore")
        X_test_model = X_test.drop(columns=[atributo_sensivel], errors="ignore")
        sufixo = "unaware"

    resultados = []

    print("\n" + "=" * 80)
    print(f"Teste de lambdas FAGTB | {nome_experimento} | {sufixo}")
    print("=" * 80)

    for lambda_ in lambdas:

        print("\n" + "-" * 80)
        print(f"Treinando lambda = {lambda_}")
        print("-" * 80)

        # Usa exatamente o mesmo wrapper do fluxo principal.
        # Os hiperparâmetros ficam nos padrões de modelo_fagtb.
        modelo = modelo_fagtb(
            X_train=X_fit_model,
            y_train=y_fit.values,
            sensitive_train=s_fit.values,
            X_test=X_val_model,
            y_test=y_val.values,
            sensitive_test=s_val.values,
            lambda_fagtb=lambda_
        )

        y_pred_val = modelo.predict(X_val_model)

        metricas = avaliar_modelo(
            y_true=y_val.values,
            y_pred=y_pred_val,
            sensitive=s_val.values,
            grupo_privilegiado=grupo_privilegiado,
            grupo_desprivilegiado=grupo_desprivilegiado
        )

        linha = {
            "dataset_config": nome_experimento,
            "cenario": sufixo,
            "split": "validacao",
            "lambda": lambda_,
            **metricas
        }

        resultados.append(linha)

        print(
            f"lambda={lambda_} | "
            f"acc={metricas['accuracy']:.4f} | "
            f"bal_acc={metricas['balanced_accuracy']:.4f} | "
            f"f1={metricas['f1']:.4f} | "
            f"EOdds={metricas['equalized_odds']:.4f} | "
            f"EOp={metricas['equal_opportunity']:.4f} | "
            f"p-rule={metricas['p_rule']:.2f}"
        )

    df_resultados = pd.DataFrame(resultados)

    df_val = df_resultados[
        df_resultados["split"] == "validacao"
    ].copy()

    melhor_lambda, info_escolha, df_val_diagnostico = (
        escolher_lambda_antes_despencar(
            df_val=df_val,
            metrica_desempenho=metrica_despencar,
            metrica_justica=metrica_justica,
            queda_brusca_minima=queda_brusca_minima
        )
    )

    print("\n" + "=" * 80)
    print("Escolha automática")
    print("=" * 80)
    print(f"Critério: {info_escolha['criterio']}")
    print(f"Métrica para detectar queda: {metrica_despencar}")
    print(f"Queda brusca mínima: {queda_brusca_minima}")
    print(f"Métrica de justiça fallback: {metrica_justica}")
    print(f"Melhor lambda escolhido: {melhor_lambda}")

    if info_escolha["lambda_onde_caiu"] is not None:
        print(
            f"Queda detectada de {info_escolha['lambda_antes_queda']} "
            f"para {info_escolha['lambda_onde_caiu']}: "
            f"{info_escolha['valor_antes_queda']:.4f} -> "
            f"{info_escolha['valor_depois_queda']:.4f} "
            f"(queda = {info_escolha['queda_detectada']:.4f})"
        )

    caminho_diag = (
        pasta_saida /
        f"{nome_experimento}_{sufixo}_diagnostico_quedas.csv"
    )
    df_val_diagnostico.to_csv(caminho_diag, index=False)

    print("\n" + "=" * 80)
    print(f"Treinando modelo final com lambda = {melhor_lambda}")
    print("=" * 80)

    modelo_final = modelo_fagtb(
        X_train=X_train_model,
        y_train=y_train.values,
        sensitive_train=s_train.values,
        X_test=X_test_model,
        y_test=y_test.values,
        sensitive_test=s_test.values,
        lambda_fagtb=melhor_lambda
    )

    y_pred_test = modelo_final.predict(X_test_model)

    metricas_teste = avaliar_modelo(
        y_true=y_test.values,
        y_pred=y_pred_test,
        sensitive=s_test.values,
        grupo_privilegiado=grupo_privilegiado,
        grupo_desprivilegiado=grupo_desprivilegiado
    )

    linha_teste = {
        "dataset_config": nome_experimento,
        "cenario": sufixo,
        "split": "teste_final",
        "lambda": melhor_lambda,
        **metricas_teste
    }

    df_resultados = pd.concat(
        [
            df_resultados,
            pd.DataFrame([linha_teste])
        ],
        ignore_index=True
    )

    df_resultados["lambda_escolhido"] = melhor_lambda
    df_resultados["criterio_escolha"] = info_escolha["criterio"]
    df_resultados["metrica_despencar"] = metrica_despencar
    df_resultados["queda_brusca_minima"] = queda_brusca_minima

    print("\nResultado final no teste:")
    print(
        f"lambda={melhor_lambda} | "
        f"acc={metricas_teste['accuracy']:.4f} | "
        f"bal_acc={metricas_teste['balanced_accuracy']:.4f} | "
        f"f1={metricas_teste['f1']:.4f} | "
        f"EOdds={metricas_teste['equalized_odds']:.4f} | "
        f"EOp={metricas_teste['equal_opportunity']:.4f} | "
        f"p-rule={metricas_teste['p_rule']:.2f}"
    )

    caminho_figura = None

    if gerar_grafico:
        caminho_figura = plotar_tradeoff_lambda_duplo_eixo(
            df_resultados=df_resultados,
            melhor_lambda=melhor_lambda,
            info_escolha=info_escolha,
            nome_experimento=nome_experimento,
            cenario=sufixo,
            pasta_figuras=pasta_saida / "figuras",
            salvar_figura=True,
            mostrar_figura=mostrar_grafico,
            dpi=200
        )

    df_resultados["caminho_figura"] = (
        str(caminho_figura) if caminho_figura is not None else None
    )

    caminho_csv = (
        pasta_saida /
        f"{nome_experimento}_{sufixo}_lambdas.csv"
    )

    df_resultados.to_csv(caminho_csv, index=False)

    caminho_modelo = None

    if salvar_modelo_final:
        caminho_modelo = (
            pasta_saida /
            f"{nome_experimento}_{sufixo}_lambda_{melhor_lambda}.joblib"
        )

        joblib.dump(modelo_final, caminho_modelo)

    print(f"Resultados salvos em: {caminho_csv}")
    print(f"Diagnóstico das quedas salvo em: {caminho_diag}")

    if caminho_figura is not None:
        print(f"Figura salva em: {caminho_figura}")

    if caminho_modelo is not None:
        print(f"Modelo final salvo em: {caminho_modelo}")

    return df_resultados, modelo_final, melhor_lambda, info_escolha

In [63]:
# ============================================================
# Rodar busca de lambda para todos os datasets/configs
# usando os splits salvos pelo fluxo principal
# ============================================================

resumos_lambdas = []
resultados_detalhados = []

for caminho in tqdm(arquivos, desc="Datasets", unit="dataset"):

    if "compas" in caminho.name.lower():
        print(f"\nPulando dataset COMPAS: {caminho.name}")
        continue
    
    nome_dataset = caminho.stem.lower()
    print(f"\nDataset: {nome_dataset}")

    if nome_dataset not in configs:
        print(f"Dataset não encontrado em configs_datasets.json: {nome_dataset}")
        continue

    config_dataset = configs[nome_dataset]
    configs_sensiveis = config_dataset["configs_sensiveis"]

    X_train, X_test, y_train, y_test = carregar_splits_salvos(
        nome_dataset=nome_dataset,
        pasta_splits=pasta_splits
    )

    for config in configs_sensiveis:

        nome_config = config["nome"]
        atributo_sensivel = config["atributo_sensivel"]

        print(f"\nConfig sensível: {nome_config}")

        if atributo_sensivel not in X_train.columns:
            print(f"Atributo sensível não encontrado em X_train: {atributo_sensivel}")
            continue

        nome_experimento = f"{nome_dataset}_{nome_config}"

        caminho_resultado = (
            pasta_testes_fagtb /
            f"{nome_experimento}_aware_lambdas.csv"
        )

        if caminho_resultado.exists() and not sobrescrever:
            print(f"Resultado já existe. Carregando: {caminho_resultado}")

            df_lambdas = pd.read_csv(caminho_resultado)
            df_teste = df_lambdas[df_lambdas["split"] == "teste_final"]

            if len(df_teste) > 0:
                melhor_lambda = df_teste["lambda"].iloc[0]
            else:
                melhor_lambda = df_lambdas["lambda_escolhido"].dropna().iloc[0]

            info_escolha = {
                "criterio": df_lambdas["criterio_escolha"].dropna().iloc[0],
                "metrica_desempenho": metrica_despencar,
                "metrica_justica": metrica_justica,
                "queda_brusca_minima": queda_brusca_minima,
                "lambda_antes_queda": None,
                "lambda_onde_caiu": None,
                "queda_detectada": None,
                "valor_antes_queda": None,
                "valor_depois_queda": None
            }

            modelo_final = None

        else:
            df_lambdas, modelo_final, melhor_lambda, info_escolha = testar_lambdas_fagtb(
                X_train=X_train,
                X_test=X_test,
                y_train=y_train,
                y_test=y_test,

                atributo_sensivel=atributo_sensivel,
                grupo_privilegiado=config["grupo_privilegiado"],
                grupo_desprivilegiado=config["grupo_desprivilegiado"],

                lambdas=lambdas,
                random_state=random_state,
                test_size_validacao=test_size_validacao,

                metrica_despencar=metrica_despencar,
                queda_brusca_minima=queda_brusca_minima,
                metrica_justica=metrica_justica,

                pasta_saida=pasta_testes_fagtb,
                nome_experimento=nome_experimento,

                usar_atributo_sensivel_no_modelo=True,
                salvar_modelo_final=salvar_modelo_final_teste,
                gerar_grafico=gerar_grafico,
                mostrar_grafico=mostrar_grafico
            )

        df_lambdas = df_lambdas.copy()
        df_lambdas["dataset"] = nome_dataset
        df_lambdas["config_sensivel"] = nome_config
        df_lambdas["atributo_sensivel"] = atributo_sensivel

        resultados_detalhados.append(df_lambdas)

        resumos_lambdas.append({
            "dataset": nome_dataset,
            "config_sensivel": nome_config,
            "atributo_sensivel": atributo_sensivel,
            "grupo_privilegiado": config["grupo_privilegiado"],
            "grupo_desprivilegiado": config["grupo_desprivilegiado"],
            "lambda_fagtb_sugerido": float(melhor_lambda),
            "criterio_escolha": info_escolha["criterio"],
            "metrica_despencar": metrica_despencar,
            "queda_brusca_minima": queda_brusca_minima,
            "metrica_justica": metrica_justica,
            "lambda_atual_no_json": config.get("lambda_fagtb", None),
            "lambda_onde_caiu": info_escolha.get("lambda_onde_caiu"),
            "queda_detectada": info_escolha.get("queda_detectada"),
            "valor_antes_queda": info_escolha.get("valor_antes_queda"),
            "valor_depois_queda": info_escolha.get("valor_depois_queda")
        })

df_resumo_lambdas = pd.DataFrame(resumos_lambdas)

caminho_resumo = pasta_testes_fagtb / "resumo_lambdas_sugeridos.csv"
df_resumo_lambdas.to_csv(caminho_resumo, index=False)

display(df_resumo_lambdas)

print(f"Resumo salvo em: {caminho_resumo}")

if len(resultados_detalhados) > 0:
    df_resultados_detalhados = pd.concat(resultados_detalhados, ignore_index=True)

    caminho_detalhado = pasta_testes_fagtb / "resultados_detalhados_lambdas.csv"
    df_resultados_detalhados.to_csv(caminho_detalhado, index=False)

    print(f"Resultados detalhados salvos em: {caminho_detalhado}")

Datasets:   0%|          | 0/2 [00:00<?, ?dataset/s]


Pulando dataset COMPAS: compas.csv

Dataset: hmda

Config sensível: african_american_0

Teste de lambdas FAGTB | hmda_african_american_0 | aware

--------------------------------------------------------------------------------
Treinando lambda = 0.001
--------------------------------------------------------------------------------
0 25436.093854972147 9445.513909347184 9470.950003202157 Accuracy: 0.7812  test :  0.7811  Prule Train :  1.0  Prule test :  1.0
5 11525.37025523724 8860.229981449635 8871.755351704871 Accuracy: 0.7812  test :  0.7811  Prule Train :  1.0  Prule test :  1.0
10 6976.1332880312875 8376.148506747471 8383.124640035505 Accuracy: 0.7812  test :  0.7811  Prule Train :  1.0  Prule test :  1.0
15 48401.12676582995 7972.092740002212 8020.493866768043 Accuracy: 0.7812  test :  0.7811  Prule Train :  1.0  Prule test :  1.0
20 75917.05446098655 7640.078876167079 7715.995930628065 Accuracy: 0.8384  test :  0.8293  Prule Train :  0.968046101566622  Prule test :  0.969991193

Datasets: 100%|██████████| 2/2 [11:30<00:00, 345.25s/dataset]

Figura salva em: resultados\testes_lambda_fagtb\figuras\hmda_caucasian_1_aware_tradeoff_lambda.png
Resultados salvos em: resultados\testes_lambda_fagtb\hmda_caucasian_1_aware_lambdas.csv
Diagnóstico das quedas salvo em: resultados\testes_lambda_fagtb\hmda_caucasian_1_aware_diagnostico_quedas.csv
Figura salva em: resultados\testes_lambda_fagtb\figuras\hmda_caucasian_1_aware_tradeoff_lambda.png


,dataset,config_sensivel,atributo_sensivel,grupo_privilegiado,grupo_desprivilegiado,lambda_fagtb_sugerido,criterio_escolha,metrica_despencar,queda_brusca_minima,metrica_justica,lambda_atual_no_json,lambda_onde_caiu,queda_detectada,valor_antes_queda,valor_depois_queda
0,hmda,african_american_0,Applicant_African-American,0,1,0.03,antes_da_primeira_queda_brusca,balanced_accuracy,0.05,equalized_odds,0.01,0.1,0.053247,0.679887,0.626640
1,hmda,caucasian_1,Applicant_Caucasian,1,0,0.03,antes_da_primeira_queda_brusca,balanced_accuracy,0.05,equalized_odds,0.01,0.1,0.106310,0.678744,0.572434


Resumo salvo em: resultados\testes_lambda_fagtb\resumo_lambdas_sugeridos.csv
Resultados detalhados salvos em: resultados\testes_lambda_fagtb\resultados_detalhados_lambdas.csv
